# 01 - Preprocess parcellated signal and tokenize

Run the parcel data through BrainOmni's tokeniser using the geometry from notebook 00, producing
the `proj_out` arrays the transformer blocks consume.

- The chain inside BrainOmni is **tokenize → add neuro embeddings → project → transformer blocks**. `proj_out` is captured after the projection and before the blocks, matching the existing `proj_out*.npy` files so everything downstream works unchanged.
- Preprocessing similar to `src/preprocessing.py`:
    - Resample 600 → 256 Hz.
    - Z-normalise per channel.
    - Take two non-overlapping 30 second segments, matching BrainOmni's 30 s pretraining context.
- Reads one subject at a time. `tsAD.mat` is ~5.5 GB and `tsCONT.mat` ~6 GB, and both are HDF5 (MATLAB v7.3), so h5py reads lazily rather than loading either in full.

In [14]:
import os, sys, json
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np, h5py, torch
import scipy.signal
sys.path.insert(0, '/Users/faenegoro/Documents/Neuroimaging/BrainOmni')
from brainomni.model import BrainOmni

DATA = '../data'; OUT = '../results_parcel'
CKPT_ROOT = '/Users/faenegoro/Documents/Neuroimaging/BrainOmni/ckpt_collection'

MODEL   = 'tiny'         # 'tiny' or 'base'
SUFFIX  = 'cortical'     # matches what notebook 00 saved
FS_ORIG, FS_TARGET, SEGMENT_SECS = 600, 256, 30
SEG_LEN = SEGMENT_SECS * FS_TARGET     # 7680 samples per segment

device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
print('device:', device, '| model:', MODEL)

device: mps | model: tiny


In [15]:
# Geometry from notebook 00. keep_idx selects the cortical rows out of Pooja's 246.
L        = np.load(f'{OUT}/parcel_L_{SUFFIX}.npy')          # (C, 6) positions + orientations
S        = np.load(f'{OUT}/parcel_S_{SUFFIX}.npy')          # (C,)   fabricated sensor types
keep_idx = np.load(f'{OUT}/parcel_keep_idx_{SUFFIX}.npy')   # (C,)   rows to take from the signal
n_ch = len(L)
print(f'geometry: L {L.shape} | S {S.shape} | keeping {n_ch} of 246 rows')

pos_t   = torch.tensor(L, dtype=torch.float32).unsqueeze(0).to(device)
stype_t = torch.tensor(S, dtype=torch.int32).unsqueeze(0).to(device)

geometry: L (210, 6) | S (210,) | keeping 210 of 246 rows


In [16]:
cfg  = json.load(open(f'{CKPT_ROOT}/{MODEL}/model_cfg.json'))
ckpt = torch.load(f'{CKPT_ROOT}/{MODEL}/BrainOmni.pt', map_location='cpu', weights_only=True)
model = BrainOmni(**cfg)
model.load_state_dict(ckpt, strict=False)
for p in model.parameters():
    p.requires_grad = False
model = model.to(device).eval()

def project_only(x, pos, stype):
    """Reproduce BrainOmni.encode up to the projection, stopping before the transformer blocks.
    This is exactly what the existing proj_out*.npy files contain."""
    with torch.no_grad():
        tok, _ = model.tokenizer.tokenize(x, pos, stype, model.overlap_ratio)
        B, Cc, W, _ = tok.shape
        neuro = model.tokenizer.encoder.neuros.type_as(tok).detach().view(1, Cc, 1, -1)
        return model.projection(tok + neuro)

print('model loaded, frozen.')

model loaded, frozen.


In [17]:
def preprocess(sig):
    """(C, 36000) at 600 Hz -> two (C, 7680) segments at 256 Hz, z-normalized per channel."""
    sig = scipy.signal.resample_poly(sig, up=32, down=75, axis=1).astype(np.float32)
    mean = sig.mean(axis=1, keepdims=True)
    std  = sig.std(axis=1, keepdims=True) + 1e-13
    sig  = (sig - mean) / std
    return sig[:, :SEG_LEN], sig[:, SEG_LEN:SEG_LEN*2]

def tokenize_file(path, label):
    """Stream one .mat file subject by subject. Returns proj_out for both segments, plus ids."""
    out1, out2, ids = [], [], []
    with h5py.File(path, 'r') as f:
        n_subj = f['matrix'].shape[2]        # HDF5 stores it reversed: (time, region, subject)
        refs = f['radid'][:].flatten()
        for i in range(n_subj):
            sig = np.array(f['matrix'][:, :, i]).T      # -> (246, 36000)
            sig = sig[keep_idx]                         # -> (C, 36000)
            s1, s2 = preprocess(sig)
            for seg, store in ((s1, out1), (s2, out2)):
                x = torch.tensor(seg, dtype=torch.float32).unsqueeze(0).to(device)
                store.append(project_only(x, pos_t, stype_t).squeeze(0).cpu().numpy())
            ids.append(''.join(chr(c) for c in np.asarray(f[refs[i]]).flatten()))
            if (i + 1) % 10 == 0:
                print(f'  {os.path.basename(path)}: {i+1}/{n_subj}', flush=True)
    return np.stack(out1), np.stack(out2), ids, [label] * n_subj

print('helpers ready.')

helpers ready.


In [18]:
# AD = 1, Control = 0. A few minutes per file; progress prints every 10 subjects.
import time
t0 = time.time()
ad1, ad2, ad_ids, ad_lab = tokenize_file(f'{DATA}/ts48/tsAD.mat',   1)
print(f'AD done: {ad1.shape} ({(time.time()-t0)/60:.1f} min)', flush=True)

ct1, ct2, ct_ids, ct_lab = tokenize_file(f'{DATA}/ts48/tsCONT.mat', 0)
print(f'CONT done: {ct1.shape} ({(time.time()-t0)/60:.1f} min total)', flush=True)

  tsAD.mat: 10/77
  tsAD.mat: 20/77
  tsAD.mat: 30/77
  tsAD.mat: 40/77
  tsAD.mat: 50/77
  tsAD.mat: 60/77
  tsAD.mat: 70/77
AD done: (77, 16, 160, 256) (40.9 min)
  tsCONT.mat: 10/84
  tsCONT.mat: 20/84
  tsCONT.mat: 30/84
  tsCONT.mat: 40/84
  tsCONT.mat: 50/84
  tsCONT.mat: 60/84
  tsCONT.mat: 70/84
  tsCONT.mat: 80/84
CONT done: (84, 16, 160, 256) (104.1 min total)


In [19]:
proj1  = np.concatenate([ad1, ct1]).astype(np.float32)
proj2  = np.concatenate([ad2, ct2]).astype(np.float32)
labels = np.array(ad_lab + ct_lab, dtype=np.int64)
subj   = np.array(ad_ids + ct_ids)

print('proj_out1:', proj1.shape, '| proj_out2:', proj2.shape)
print('labels:', labels.shape, '| AD', int((labels==1).sum()), '| Control', int((labels==0).sum()))
print('finite values only?', bool(np.isfinite(proj1).all() and np.isfinite(proj2).all()))
print('value range:', float(proj1.min()), 'to', float(proj1.max()))

np.save(f'{OUT}/parcel_proj_out1_{MODEL}_{SUFFIX}.npy', proj1)
np.save(f'{OUT}/parcel_proj_out2_{MODEL}_{SUFFIX}.npy', proj2)
np.save(f'{OUT}/parcel_labels_{SUFFIX}.npy', labels)
np.save(f'{OUT}/parcel_subject_ids_{SUFFIX}.npy', subj)
print(f'\nsaved to {OUT}/')

proj_out1: (161, 16, 160, 256) | proj_out2: (161, 16, 160, 256)
labels: (161,) | AD 77 | Control 84
finite values only? True
value range: -3.327484130859375 to 3.82293701171875

saved to ../results_parcel/
